In [ ]:


!pip install openai langchain langchain-openai requests -q

In [ ]:


import os
import json
import requests
from datetime import datetime
from langchain_openai import ChatOpenAI
from langchain.agents import tool, create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

stream_handler = StreamingStdOutCallbackHandler()

llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    streaming=True,
    callbacks=[stream_handler],
    request_timeout=600,
    temperature=0
)

print("✓ Modelo configurado con streaming habilitado")
print(f"Modelo: {llm.model_name}")
print(f"Streaming: {llm.streaming}")

In [ ]:


CENTROS = {
    "ensenada": {"lat": -41.140459, "lon": -72.404236, "nombre": "Piscicultura Petrohué"},
    "puelche":  {"lat": -41.733,    "lon": -73.602,    "nombre": "Centro Puelche"},
    "huito":    {"lat": -41.783,    "lon": -73.583,    "nombre": "Centro Huito (San José)"}
}

@tool
def get_clima_actual(centro: str) -> str:
    """Obtiene el clima actual para un centro de cultivo de Camanchaca.
    El parámetro centro puede ser: ensenada, puelche o huito."""
    if centro.lower() not in CENTROS:
        return f"Centro '{centro}' no encontrado. Opciones: ensenada, puelche, huito."
    
    datos = CENTROS[centro.lower()]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={datos['lat']}&longitude={datos['lon']}"
        f"&current=temperature_2m,wind_speed_10m,precipitation,weathercode"
        f"&timezone=America/Santiago"
    )
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        current = data["current"]
        
        temp   = current["temperature_2m"]
        viento = current["wind_speed_10m"]
        lluvia = current["precipitation"]
        codigo = current["weathercode"]
        
        condicion = "Despejado" if codigo < 3 else "Nublado" if codigo < 50 else "Lluvia"
        
        return (
            f"Centro: {datos['nombre']}\n"
            f"Temperatura: {temp}°C\n"
            f"Viento: {viento} km/h\n"
            f"Precipitación: {lluvia} mm\n"
            f"Condición: {condicion}"
        )
    except Exception as e:
        return f"Error al obtener datos climáticos: {e}"


@tool
def get_pronostico_semana(centro: str) -> str:
    """Obtiene el pronóstico climático de 7 días para un centro de cultivo de Camanchaca.
    El parámetro centro puede ser: ensenada, puelche o huito."""
    if centro.lower() not in CENTROS:
        return f"Centro '{centro}' no encontrado. Opciones: ensenada, puelche, huito."
    
    datos = CENTROS[centro.lower()]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={datos['lat']}&longitude={datos['lon']}"
        f"&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,weathercode"
        f"&timezone=America/Santiago"
    )
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        daily = data["daily"]
        
        resultado = f"Pronóstico 7 días - {datos['nombre']}:\n"
        for i in range(7):
            fecha   = daily["time"][i]
            tmax    = daily["temperature_2m_max"][i]
            tmin    = daily["temperature_2m_min"][i]
            lluvia  = daily["precipitation_sum"][i]
            viento  = daily["wind_speed_10m_max"][i]
            codigo  = daily["weathercode"][i]
            condicion = "Despejado" if codigo < 3 else "Nublado" if codigo < 50 else "Lluvia"
            
            resultado += (
                f"\n{fecha}: {tmin}°C - {tmax}°C | "
                f"Viento: {viento} km/h | "
                f"Lluvia: {lluvia} mm | {condicion}"
            )
        return resultado
    except Exception as e:
        return f"Error al obtener pronóstico: {e}"


@tool
def evaluar_operacion(centro: str, operacion: str) -> str:
    """Evalúa si las condiciones climáticas son seguras para realizar una operación en Camanchaca.
    centro: ensenada, puelche o huito.
    operacion: cosecha, biometría o tratamiento."""
    if centro.lower() not in CENTROS:
        return f"Centro '{centro}' no encontrado."
    
    datos = CENTROS[centro.lower()]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={datos['lat']}&longitude={datos['lon']}"
        f"&current=temperature_2m,wind_speed_10m,precipitation,weathercode"
        f"&timezone=America/Santiago"
    )
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        current = data["current"]
        
        viento = current["wind_speed_10m"]
        lluvia = current["precipitation"]
        temp   = current["temperature_2m"]
        
        alertas = []
        if viento > 40:
            alertas.append(f" Viento peligroso: {viento} km/h (límite: 40 km/h)")
        if lluvia > 10:
            alertas.append(f" Lluvia intensa: {lluvia} mm")
        if temp < 5:
            alertas.append(f" Temperatura muy baja: {temp}°C")
        if temp > 18:
            alertas.append(f" Temperatura elevada: {temp}°C (riesgo para FCR)")
        
        if not alertas:
            return f" Condiciones APTAS para {operacion} en {datos['nombre']}."
        else:
            return f" Condiciones NO APTAS para {operacion} en {datos['nombre']}:\n" + "\n".join(alertas)
    except Exception as e:
        return f"Error al evaluar condiciones: {e}"


tools = [get_clima_actual, get_pronostico_semana, evaluar_operacion]

print("✓ Herramientas climáticas definidas.")
print(f"  Herramientas: {[t.name for t in tools]}")

In [ ]:


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Eres un asistente experto en acuicultura para Salmones Camanchaca. "
        "Monitoreas las condiciones climáticas de los centros Ensenada, Puelche y Huito "
        "en la región de Los Lagos, Chile, y apoyas decisiones operativas del equipo. "
        "Recuerdas el contexto de la conversación para dar respuestas coherentes."
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

agent          = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✓ Agente y AgentExecutor listos.")

In [ ]:


from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

print("=== MEMORIA MANUAL - CONVERSACIÓN CON EL OPERADOR ===\n")

print("1. Primera pregunta:")
query1 = "¿Cuál es el clima actual en el centro Ensenada?"
response1 = agent_executor.invoke({
    "input": query1,
    "chat_history": chat_history
})
print(f"\nRespuesta: {response1['output']}\n")

chat_history.append(HumanMessage(content=query1))
chat_history.append(AIMessage(content=response1["output"]))
print("Historial actualizado.\n")

print("2. Segunda pregunta (seguimiento):")
query2 = "¿Y es seguro hacer la cosecha allí hoy?"
response2 = agent_executor.invoke({
    "input": query2,
    "chat_history": chat_history
})
print(f"\nRespuesta: {response2['output']}\n")

chat_history.append(HumanMessage(content=query2))
chat_history.append(AIMessage(content=response2["output"]))
print("Historial actualizado.\n")

print("3. Tercera pregunta (seguimiento):")
query3 = "¿De qué centro me estabas hablando?"
response3 = agent_executor.invoke({
    "input": query3,
    "chat_history": chat_history
})
print(f"\nRespuesta: {response3['output']}\n")

print("=== CONTENIDO DE LA MEMORIA ===")
for msg in chat_history:
    tipo = " Operador" if isinstance(msg, HumanMessage) else " Agente"
    print(f"{tipo}: {msg.content[:80]}...")

In [ ]:


from langchain.memory import ConversationBufferMemory

memory_buffer = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

def chat_buffer(query: str):
    history = memory_buffer.load_memory_variables({})["chat_history"]
    response = agent_executor.invoke({
        "input": query,
        "chat_history": history
    })
    memory_buffer.save_context(
        {"input": query},
        {"output": response["output"]}
    )
    return response["output"]

print("=== CONVERSATION BUFFER MEMORY ===")
print("Mantiene el historial completo de la conversación\n")

print("1. Primera pregunta:")
r1 = chat_buffer("¿Cómo está el clima en Puelche ahora?")
print(f"Respuesta: {r1}\n")

print("2. Segunda pregunta (seguimiento):")
r2 = chat_buffer("¿Hay riesgo de que afecte la biometría programada?")
print(f"Respuesta: {r2}\n")

print("=== ESTADO DE LA MEMORIA ===")
history = memory_buffer.load_memory_variables({})["chat_history"]
print(f"Total de mensajes almacenados: {len(history)}")
for i, msg in enumerate(history, 1):
    print(f"{i}. {msg.type}: {msg.content[:60]}...")

In [ ]:


from langchain.memory import ConversationBufferWindowMemory

memory_window = ConversationBufferWindowMemory(
    k=2,
    memory_key="chat_history",
    return_messages=True
)

def chat_window(query: str):
    history = memory_window.load_memory_variables({})["chat_history"]
    response = agent_executor.invoke({
        "input": query,
        "chat_history": history
    })
    memory_window.save_context(
        {"input": query},
        {"output": response["output"]}
    )
    return response["output"]

print("=== CONVERSATION BUFFER WINDOW MEMORY (k=2) ===")
print("Solo recuerda los últimos 2 intercambios\n")

interacciones = [
    "¿Cuál es el clima en Ensenada?",
    "¿Y en Puelche?",
    "¿Y en Huito?",
    "¿De qué centro me hablaste primero?"
]

for i, query in enumerate(interacciones, 1):
    print(f"{'='*20} INTERACCIÓN {i} {'='*20}")
    print(f" Operador: {query}")
    respuesta = chat_window(query)
    
    history      = memory_window.load_memory_variables({})["chat_history"]
    total        = len(memory_window.chat_memory.messages)
    visible      = len(history)
    
    print(f"\n ESTADO DE LA MEMORIA:")
    print(f"   Total almacenado: {total} mensajes")
    print(f"   Visible al modelo: {visible} mensajes")
    print(f"   Descartados: {total - visible} mensajes\n")

In [ ]:


from langchain.memory import ConversationSummaryMemory

memory_summary = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    return_messages=True
)

def chat_summary(query: str):
    history = memory_summary.load_memory_variables({})["chat_history"]
    response = agent_executor.invoke({
        "input": query,
        "chat_history": history
    })
    memory_summary.save_context(
        {"input": query},
        {"output": response["output"]}
    )
    return response["output"]

print("=== CONVERSATION SUMMARY MEMORY ===")
print("Resume conversaciones largas para ahorrar tokens\n")

consultas = [
    "Soy Carlos, jefe del centro Ensenada. ¿Cómo está el clima hoy?",
    "¿Es seguro programar la cosecha para mañana?",
    "¿Y qué hay del pronóstico para el resto de la semana?",
    "¿Sobre qué estábamos hablando y qué decisiones tomamos?"
]

for i, query in enumerate(consultas, 1):
    print(f"{'='*15} INTERACCIÓN {i} {'='*15}")
    print(f" Operador: {query}")
    respuesta = chat_summary(query)
    
    history      = memory_summary.load_memory_variables({})["chat_history"]
    tiene_resumen = any(
        "resumen" in str(msg).lower() or "summary" in str(msg).lower()
        for msg in history
    )
    
    print(f"\n ESTADO DE LA MEMORIA:")
    print(f"   Mensajes en contexto: {len(history)}")
    print(f"   Tiene resumen: {' Sí' if tiene_resumen else ' No'}\n")

print("\n=== COMPARACIÓN DE ESTRATEGIAS ===")
print("Buffer Memory:        Historial completo. Ideal para conversaciones cortas.")
print("Window Memory (k=2):  Solo últimos 2 intercambios. Balance costo/contexto.")
print("Summary Memory:       Resume el historial. Ideal para conversaciones largas.")

In [ ]:


from langchain.memory import ConversationBufferMemory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

history_store = {}

def get_session_history(session_id: str):
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

prompt_session = ChatPromptTemplate.from_messages([
    (
        "system",
        "Eres un asistente experto en acuicultura para Salmones Camanchaca. "
        "Apoyas al equipo operativo con información climática en tiempo real "
        "para los centros Ensenada, Puelche y Huito en Los Lagos, Chile."
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

agent_session    = create_openai_tools_agent(llm, tools, prompt_session)
executor_session = AgentExecutor(agent=agent_session, tools=tools, verbose=False)

conversation = RunnableWithMessageHistory(
    executor_session,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

print("=== SIMULACIÓN JORNADA OPERATIVA CAMANCHACA ===\n")

session_id = "turno_manana_001"

jornada = [
    ("Carlos - Jefe Ensenada",  "Buenos días. ¿Cómo están las condiciones en Ensenada para hoy?"),
    ("Carlos - Jefe Ensenada",  "¿Podemos proceder con la cosecha programada para esta mañana?"),
    ("María - Jefe Puelche",    "Hola, me acabo de unir. ¿Qué tal el clima en Puelche esta semana?"),
    ("Carlos - Jefe Ensenada",  "¿Recuerdas qué operación tenía programada yo esta mañana?"),
]

for operador, consulta in jornada:
    print(f" {operador}: {consulta}")
    response = conversation.invoke(
        {"input": consulta},
        config={"configurable": {"session_id": session_id}}
    )
    print(f" Agente: {response['output']}\n")

print("=== HISTORIAL DE LA SESIÓN ===")
historial = history_store[session_id].messages
print(f"Total de mensajes: {len(historial)}")
for i, msg in enumerate(historial, 1):
    rol = " Operador" if msg.type == "human" else " Agente"
    print(f"{i}. {rol}: {msg.content[:70]}...")